In [135]:
from typing import Any

import altair as alt
import pandas as pd

import json
from pathlib import Path

# Altair stores the input data in all visualizations, and we're not being careful about the size of the data we're passing.
# If you want to export the visualizations and embed on the web, you might want to comment out this line and look into:
# https://altair-viz.github.io/user_guide/large_datasets.html#vegafusion-data-transformer
alt.data_transformers.disable_max_rows()

# Plug in filepaths to OpenHands evaluation data here -- anything produced using the OpenHands SWE-bench evaluation framework
# should be compatible.
filepaths = [
    Path("/Users/calvin/all-hands/data/condenser-costs/no-condenser"),
    Path("/Users/calvin/all-hands/data/condenser-costs/structured-summary-fix"),
    Path("/Users/calvin/all-hands/data/condenser-costs/summary-prompt-improvements-fix"),
]
    

In [136]:

def get_completions_from_dir(completions_dir: Path) -> list[dict[str, Any]]:
    completions = []
    for file in completions_dir.iterdir():
        if file.is_file() and file.suffix == ".json":
            with open(file, "r") as f:
                completions.append(json.load(f))
    return completions

def get_completions_from_eval(eval_dir: Path) -> dict[str, list[dict[str, Any]]]:
    completions = {}
    for file in (eval_dir / "llm_completions").iterdir():
        if file.is_dir():
            completions[file.name] = get_completions_from_dir(file)
    return completions


In [137]:
rows = []
for filepath in filepaths:
    completions = get_completions_from_eval(Path(filepath))
    for instance_id, completion_dicts in completions.items():
        # Sort completions by timestamp
        completion_dicts.sort(key=lambda x: x["timestamp"])

        # Build the rows
        for index, completion in enumerate(completion_dicts):
            row = {
                "experiment": filepath.name,
                "instance_id": instance_id,
                "index": index,
                "cost": completion["cost"],
            }
            rows.append(row)

df = pd.DataFrame(rows).pivot_table(
    index=["instance_id"],
    columns="experiment",
    values="cost",
    aggfunc="sum",
)

In [138]:
def average_cost(df: pd.DataFrame, experiment: str) -> float:
    costs = (df["no-condenser"] - df[experiment]).tolist()
    return sum(costs) / len(costs)

for filepath in filepaths:
    experiment = filepath.name
    avg_cost = average_cost(df, experiment)
    print(f"Average savings for {experiment}: {avg_cost:.2f}")

Average savings for no-condenser: 0.00
Average savings for structured-summary-fix: -0.09
Average savings for summary-prompt-improvements-fix: -0.07


In [145]:
# Find indices of outliers
def find_outliers_zscore(series, threshold=3):
    z_scores = (series - series.mean()) / series.std()
    return series[abs(z_scores) > threshold].index.tolist()

outliers = set(find_outliers_zscore(df["summary-prompt-improvements-fix"]) + find_outliers_zscore(df["structured-summary-fix"]))

df_wo_outliers = df.drop(index=outliers)

for filepath in filepaths:
    experiment = filepath.name
    avg_cost = average_cost(df_wo_outliers, experiment)
    print(f"Average savings for {experiment} (without outliers): {avg_cost:.2f}")

Average savings for no-condenser (without outliers): 0.00
Average savings for structured-summary-fix (without outliers): -0.04
Average savings for summary-prompt-improvements-fix (without outliers): 0.05


So there are a handful of outliers that are dramatically impacting the cost of condensation. If we can figure out why they differ so much from the other runs in the same condensation, we might be able to smooth out the performance.

In [140]:
# Load the full data for comparison
from typing import Iterable
from analysis.models.openhands import Evaluation, EvaluationOutput, SWEBenchResult
from analysis.usage import per_iteration_resource_usage
data = [Evaluation.from_filepath(str(filepath)) for filepath in filepaths]

def per_step(output: EvaluationOutput, result: SWEBenchResult) -> Iterable[dict[str, Any]]:
    for step, step_usage in enumerate(per_iteration_resource_usage(output)):
        yield {
            "resolved": result.test_result.report.resolved,
            **step_usage.model_dump(),
            "iteration": step / 2,
        }

usage_df = pd.concat([d.multi_to_dataframe(per_step) for d in data])
usage_df = usage_df[usage_df["instance_id"].isin(outliers)]

In [141]:
charts = []
for outlier, outlier_df in usage_df.groupby("instance_id"):
    chart = alt.Chart(outlier_df, title=outlier).mark_line().encode(
        alt.X("iteration").scale(domain=(0, 150)),
        alt.Y("prompt_tokens").scale(domain=(0, 150_000)),
        alt.Color("experiment"),
        alt.Tooltip(["iteration", "prompt_tokens"]),
    )
    charts.append(chart)

alt.hconcat(*charts)

alt.HConcatChart(...)

With the exception of `sympy__sympy-13877`, all the `no-condenser` runs are dramatically shorter than the equivalent condensed runs. That could explain some of the cost difference.

Follow up questions:
1. Does `no-condenser` resolve the instances where it finishes earlier? If so, that's a big downside for the condensers.
2. Why do costs diverge so dramatically before the first condensation event? Should be at iteration 40, but we're seeing the condensed runs more expensive as early as 15.
3. Dramatic spike upwards in `summary-prompt-improvements` around iteration 60 in `sympy__sympy-22080`?

In [142]:
rows = []
for experiment_eval in data:
    for outlier in outliers:
        result = experiment_eval.get_result(outlier)
        output = experiment_eval.get_output(outlier)

        row = {
            "experiment": experiment_eval.experiment(),
            "instance_id": outlier,
            "resolved": result.test_result.report.resolved,
            "finished": output.history[-1].get("action") == "finish",
            "iterations": len(output.history) / 2,
        }
        rows.append(row)

results_df = pd.DataFrame(rows)
results_df

,experiment,instance_id,resolved,finished,iterations
0,no-condenser,sympy__sympy-13877,False,False,151.0
1,no-condenser,sphinx-doc__sphinx-7757,False,True,123.0
2,no-condenser,sphinx-doc__sphinx-8595,False,False,5.5
3,structured-summary-fix,sympy__sympy-13877,False,True,106.0
4,structured-summary-fix,sphinx-doc__sphinx-7757,False,True,114.0
5,structured-summary-fix,sphinx-doc__sphinx-8595,True,False,148.5
6,summary-prompt-improvements-fix,sympy__sympy-13877,False,False,148.5
7,summary-prompt-improvements-fix,sphinx-doc__sphinx-7757,False,False,148.5
8,summary-prompt-improvements-fix,sphinx-doc__sphinx-8595,False,False,10.5


So `no-condenser` does not resolve any of the instances. In fact, both condenser strategies resolve `sphinx-doc__sphinx-8595`, which the baseline exits after only ~5 iterations.

In [143]:
def render_step(step: dict[str, Any]) -> str:
    match step.get('action'):
        case "change_agent_state":
            return step.get("message")
        
        case "condensation":
            return step.get("message")
        
        case "finish":
            return "<FINISH>"
        
        case "message":
            return step.get("message")
        
        case "recall":
            return "<RECALL>"
        
        case "run":
            command = step.get("message")
            thought = step["tool_call_metadata"]["model_response"]["choices"][0]["message"]["content"]
            return f"<RUN {command}>\n{thought}"
        
        case "read":
            command = step.get("message")
            thought = step["tool_call_metadata"]["model_response"]["choices"][0]["message"]["content"]
            return f"<READ {command}>\n{thought}"
    
        case "edit":
            thought = step["tool_call_metadata"]["model_response"]["choices"][0]["message"]["content"]
            return f"<EDIT>\n{thought}"

        case "think":
            command = step.get("message")
            thought = step["tool_call_metadata"]["model_response"]["choices"][0]["message"]["content"]
            return f"<THINK {command}>\n{thought}"

    if "observation" in step:
        return "<OBSERVATION>"
    
    return step.get('message')

In [144]:
# Manual inspection of the first 20 iterations to see where things go wrong
from rich.table import Table
from rich.markup import escape

outlier = "sympy__sympy-13877"

trajectories = {
    d.experiment(): d.get_output(outlier).history
    for d in data
}

table = Table(show_lines=True)
table.add_column("Iteration")
for key in trajectories.keys():
    table.add_column(key)

for i in range(60):
    row = []
    for trajectory in trajectories.values():
        if i < len(trajectory):
            row.append(render_step(trajectory[i]))
        else:
            row.append("")
    table.add_row(str(i/2), *map(escape, row))

table

┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Iteration ┃ no-condenser                    ┃ structured-summary-fix          ┃ summary-prompt-improvements-fix ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 0.0       │                                 │                                 │                                 │
│           │ <uploaded_files>                │ <uploaded_files>                │ <uploaded_files>                │
│           │ /workspace/sympy__sympy__1.1    │ /workspace/sympy__sympy__1.1    │ /workspace/sympy__sympy__1.1    │
│           │ </uploaded_files>               │ </uploaded_files>               │ </uploaded_files>               │
│           │                                 │                                 │                                 │
│           │ I've uploaded a python code     │ I've uploaded a python code     │ I've uploaded a python code     │
│           │ repository in the directory     │ repository in the directory     │ repository in the directory     │
│           │ sympy__sympy__1.1. Consider the │ sympy__sympy__1.1. Consider the │ sympy__sympy__1.1. Consider the │
│           │ following issue description:    │ following issue description:    │ following issue description:    │
│           │                                 │                                 │                                 │
│           │ <issue_description>             │ <issue_description>             │ <issue_description>             │
│           │ Matrix determinant raises       │ Matrix determinant raises       │ Matrix determinant raises       │
│           │ Invalid NaN comparison with     │ Invalid NaN comparison with     │ Invalid NaN comparison with     │
│           │ particular symbolic entries     │ particular symbolic entries     │ particular symbolic entries     │
│           │     >>> from sympy import *     │     >>> from sympy import *     │     >>> from sympy import *     │
│           │     >>> from sympy.abc import a │     >>> from sympy.abc import a │     >>> from sympy.abc import a │
│           │     >>> f = lambda n:           │     >>> f = lambda n:           │     >>> f = lambda n:           │
│           │ det(Matrix([[i + a*j for i in   │ det(Matrix([[i + a*j for i in   │ det(Matrix([[i + a*j for i in   │
│           │ range(n)] for j in range(n)]))  │ range(n)] for j in range(n)]))  │ range(n)] for j in range(n)]))  │
│           │     >>> f(1)                    │     >>> f(1)                    │     >>> f(1)                    │
│           │     0                           │     0                           │     0                           │
│           │     >>> f(2)                    │     >>> f(2)                    │     >>> f(2)                    │
│           │     -a                          │     -a                          │     -a                          │
│           │     >>> f(3)                    │     >>> f(3)                    │     >>> f(3)                    │
│           │     2*a*(a + 2) + 2*a*(2*a + 1) │     2*a*(a + 2) + 2*a*(2*a + 1) │     2*a*(a + 2) + 2*a*(2*a + 1) │
│           │ - 3*a*(2*a + 2)                 │ - 3*a*(2*a + 2)                 │ - 3*a*(2*a + 2)                 │
│           │     >>> f(4)                    │     >>> f(4)                    │     >>> f(4)                    │
│           │     0                           │     0                           │     0                           │
│           │     >>> f(5)                    │     >>> f(5)                    │     >>> f(5)                    │
│           │     nan                         │     nan                         │     nan                         │
│           │     >>> f(6)                    │     >>> f(6)                    │     >>> f(6)                    │
│           │     Traceback (most recent call │     Trac